# Testes de qualidade da Gold e do export

Testes automatizados da Gold oficial (`v_energy.gold`) e dos 12 CSVs exportados pelo `Silver_to_Gold` (DEC-030), uma seção por grupo
de teste. Atende a história D1-10 (issue #101): cada execução grava o resultado de cada teste em `v_energy.qualidade.resultados_testes`
e o notebook falha se algum teste **crítico** falhar, para o Job ficar vermelho (DEC-039).

**Como executar**
- Depois do `Silver_to_Gold` com `gravar = sim` e `exportar_csv = sim`: `Run All`.
- Cada teste roda isolado: um erro de execução vira `FALHA` (com o tipo do erro no detalhe) e os outros testes continuam.
- O **Resumo** grava todos os resultados (append, mesmo `execucao_id`) e levanta erro se algum teste `critico` tiver `FALHA`.
- Testes `alerta` (variação de linhas) nunca derrubam a execução: só ficam registrados.

**Grupos de teste**

| Seção | Teste | Tipo |
|---|---|---|
| 1 | CSV × Gold: contagem, colunas e ordem, somas de colunas numéricas, encoding (UC-13) | crítico |
| 2 | Variação de linhas contra a execução anterior (limite de 5%) | alerta |
| 3 | Reconciliações do contrato entre tabelas | crítico |
| 4 | Premissa da DEC-067: UCs de teste sem linhas nas tabelas por UC | crítico |

**Somente leitura:** o notebook só **lê** `v_energy.gold` e os CSVs do Volume e grava **apenas** em `v_energy.qualidade`. Nunca altera a
Silver, a Gold nem os CSVs.

**Confidencial:** os prints mostram só o status e o nome de cada teste; as contagens ficam na coluna `valor`, só no Databricks.
Mesmo assim, o notebook vai ao Git sem outputs (DEC-024).

## 0. Configuração
Execute esta célula antes de qualquer teste. Cria o schema `v_energy.qualidade` e a tabela de resultados, se ainda não existirem.

In [ ]:
import uuid
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql import types as T

CATALOG = "v_energy"
GOLD = f"{CATALOG}.gold"
QUALIDADE = f"{CATALOG}.qualidade"
TABELA_RESULTADOS = f"{QUALIDADE}.resultados_testes"
PASTA_CSV = f"/Volumes/{CATALOG}/gold/csv"  # mesma pasta do export do Silver_to_Gold; os CSVs nunca vão ao Git (DEC-004)
TABELAS = ["gold_dim_uc", "gold_fato_faturamento_mensal", "gold_reconciliacao_faturamento",
           "gold_continuidade_alimentador_mensal", "gold_kpis_segmento_mensal", "gold_kpis_mensais",
           "gold_receita_decomposicao", "gold_fila_suspeitas", "gold_medidor_uc", "gold_interrupcoes_uc",
           "gold_atendimentos_uc", "gold_inspecoes_uc"]

# tipos numéricos do contrato (espelho de contracts/gold_schema.yaml v0.3, como no Silver_to_Gold)
TIPOS = {"money": "decimal(14,2)", "energy_kwh": "decimal(16,3)", "integer": "int"}

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {QUALIDADE}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABELA_RESULTADOS} (
    execucao_id  STRING    COMMENT 'uuid da execução do notebook',
    executado_em TIMESTAMP COMMENT 'início da execução',
    teste        STRING    COMMENT 'nome do teste',
    tabela       STRING    COMMENT 'tabela da Gold testada',
    tipo         STRING    COMMENT 'critico ou alerta',
    status       STRING    COMMENT 'OK, FALHA, ALERTA ou LINHA_DE_BASE',
    valor        DOUBLE    COMMENT 'contagem ou diferença medida pelo teste (só no Databricks)',
    detalhe      STRING    COMMENT 'texto curto, sem dado do dataset'
) USING DELTA
COMMENT 'Resultados do notebook Testes_Qualidade_Gold: uma linha por teste e execução (append).'
""")

ESQUEMA_RESULTADOS = T.StructType([
    T.StructField("execucao_id", T.StringType()),
    T.StructField("executado_em", T.TimestampType()),
    T.StructField("teste", T.StringType()),
    T.StructField("tabela", T.StringType()),
    T.StructField("tipo", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("valor", T.DoubleType()),
    T.StructField("detalhe", T.StringType()),
])

EXECUCAO_ID = str(uuid.uuid4())
EXECUTADO_EM = datetime.now()
RESULTADOS = []  # um dicionário por teste, gravado no Resumo
CONTAGENS = {}  # contagem de cada tabela da Gold nesta execução (seções 1 e 2)


def registrar(teste, tabela, tipo, status, valor=None, detalhe=None):
    """Acumula o resultado e imprime só o status e o nome do teste (sem números)."""
    assert tipo in ("critico", "alerta"), f"tipo inválido: {tipo}"
    assert status in ("OK", "FALHA", "ALERTA", "LINHA_DE_BASE"), f"status inválido: {status}"
    RESULTADOS.append({"execucao_id": EXECUCAO_ID, "executado_em": EXECUTADO_EM, "teste": teste, "tabela": tabela,
                       "tipo": tipo, "status": status, "valor": None if valor is None else float(valor),
                       "detalhe": None if detalhe is None else str(detalhe)[:500]})
    print(f"[{status}] {teste}")


def testar(teste, tabela, tipo, funcao, *args):
    """funcao(*args) devolve (status, valor, detalhe). Erro de execução vira FALHA, e os outros testes continuam."""
    try:
        status, valor, detalhe = funcao(*args)
    except Exception as e:
        status, valor, detalhe = "FALHA", None, f"erro de execução: {type(e).__name__}"
    registrar(teste, tabela, tipo, status, valor, detalhe)


def ok_ou_falha(divergencias, detalhe_falha):
    return ("OK", 0, None) if divergencias == 0 else ("FALHA", divergencias, detalhe_falha)


def contar_tabela(nome):
    if nome not in CONTAGENS:
        CONTAGENS[nome] = spark.table(f"{GOLD}.{nome}").count()
    return CONTAGENS[nome]


print(f"Gold: {GOLD} | CSVs: {PASTA_CSV} | resultados: {TABELA_RESULTADOS} | execução {EXECUCAO_ID}")

## 1. CSV × Gold (críticos)
Para cada uma das 12 tabelas, o CSV do Volume (`<tabela>.csv`) é lido no formato do export: UTF-8 com cabeçalho, aspas dobradas e
todas as colunas como texto (`inferSchema` desligado, `multiLine` ligado). O CSV precisa ser igual à tabela gravada:

- `csv_contagem_linhas`: mesma quantidade de linhas (`valor` = CSV − tabela).
- `csv_colunas`: mesmas colunas, na mesma ordem.
- `csv_soma_<coluna>`: soma do CSV (texto convertido para o decimal ou inteiro do contrato; vazio = nulo) igual à soma da tabela,
  exata e segura com nulo (`<=>`): as duas somas nulas é `OK`, só uma nula é `FALHA`. Texto que não converte é `FALHA`. As colunas estão em `SOMAS_CSV`, só dos tipos `money`, `energy_kwh` e `integer`:
  `gold_interrupcoes_uc` e `gold_atendimentos_uc` não têm nenhuma (só `ratio`), e `gold_medidor_uc` tem só `n_trocas`.
- `csv_encoding`: nenhum valor das colunas de texto com o padrão da UC-13 (UTF-8 lido como Latin-1, ex.: `Ã¡` no lugar de `á`).

In [ ]:
SOMAS_CSV = {
    "gold_dim_uc": {"consumo_medio_kwh_12m": "energy_kwh", "energia_injetada_kwh_mes": "energy_kwh"},
    "gold_fato_faturamento_mensal": {"receita_faturada": "money", "kwh_faturado": "energy_kwh", "dias_em_atraso": "integer"},
    "gold_reconciliacao_faturamento": {"receita_faturada_total": "money", "kwh_faturado_total": "energy_kwh",
                                       "n_faturas_orfas": "integer"},
    "gold_continuidade_alimentador_mensal": {"n_uc": "integer", "n_interrupcoes": "integer"},
    "gold_kpis_segmento_mensal": {"receita_faturada": "money", "kwh_faturado": "energy_kwh", "n_uc": "integer"},
    "gold_kpis_mensais": {"receita_faturada": "money", "kwh_faturado": "energy_kwh", "n_chamados": "integer"},
    "gold_receita_decomposicao": {"receita_base": "money", "receita_atual": "money", "efeito_preco": "money"},
    "gold_fila_suspeitas": {"score_suspeita": "integer", "kwh_recuperavel_est": "energy_kwh"},
    "gold_medidor_uc": {"n_trocas": "integer"},
    "gold_interrupcoes_uc": {},
    "gold_atendimentos_uc": {},
    "gold_inspecoes_uc": {"kwh_recuperado": "energy_kwh"},
}
UC13 = r"[ÃÂ][\x{80}-\x{BF}]"  # padrão do erro de encoding da UC-13 (docs/qualidade-dados.md)
CSV_DF = {}


def ler_csv(nome):
    if nome not in CSV_DF:
        CSV_DF[nome] = (spark.read
            .option("header", "true").option("inferSchema", "false").option("encoding", "UTF-8")
            .option("escape", '"').option("multiLine", "true")
            .csv(f"{PASTA_CSV}/{nome}.csv"))
    return CSV_DF[nome]


def teste_contagem(nome):
    diferenca = ler_csv(nome).count() - contar_tabela(nome)
    return ("OK", 0, None) if diferenca == 0 else ("FALHA", diferenca, "contagem do CSV diferente da tabela")


def teste_colunas(nome):
    csv, gold = ler_csv(nome).columns, spark.table(f"{GOLD}.{nome}").columns
    if csv == gold:
        return "OK", 0, None
    faltando, sobrando = [c for c in gold if c not in csv], [c for c in csv if c not in gold]
    if faltando or sobrando:
        return "FALHA", len(faltando) + len(sobrando), f"faltando no CSV: {faltando}; sobrando no CSV: {sobrando}"
    return "FALHA", sum(a != b for a, b in zip(csv, gold, strict=True)), "mesmas colunas em outra ordem"


def teste_soma(nome, coluna, tipo):
    csv = ler_csv(nome).select(F.col(coluna).alias("texto"),
                               F.expr(f"try_cast(nullif(`{coluna}`, '') AS {TIPOS[tipo]})").alias("numero"))
    soma_gold = spark.table(f"{GOLD}.{nome}").agg(F.sum(coluna).alias("soma_gold"))
    r = (csv.agg(F.sum("numero").alias("soma_csv"),
                 F.sum((F.col("numero").isNull() & F.col("texto").isNotNull() & (F.col("texto") != "")).cast("int"))
                 .alias("invalidos"))
        .crossJoin(soma_gold)
        .select("invalidos", F.col("soma_csv").eqNullSafe(F.col("soma_gold")).alias("iguais"),
                (F.col("soma_csv") - F.col("soma_gold")).alias("diferenca"))
        .first())
    if r["invalidos"]:
        return "FALHA", r["invalidos"], f"texto do CSV que não converte para {TIPOS[tipo]}"
    if r["iguais"]:
        return "OK", 0, None
    if r["diferenca"] is None:
        return "FALHA", None, "soma nula só de um lado (CSV ou tabela)"
    return "FALHA", r["diferenca"], "soma do CSV diferente da tabela"


def teste_encoding(nome):
    tipos = dict(spark.table(f"{GOLD}.{nome}").dtypes)
    textos = [c for c in ler_csv(nome).columns if tipos.get(c) == "string" or tipos.get(c, "").startswith("array<")]
    if not textos:
        return "OK", 0, None
    r = ler_csv(nome).agg(*[F.sum(F.coalesce(F.col(c).rlike(UC13), F.lit(False)).cast("int")).alias(c)
                            for c in textos]).first().asDict()
    afetadas = [c for c, n in r.items() if n]
    return ok_ou_falha(sum(r[c] for c in afetadas), f"colunas com o padrão da UC-13: {afetadas}")


for nome in TABELAS:
    print(f"=== {nome}")
    testar("csv_contagem_linhas", nome, "critico", teste_contagem, nome)
    testar("csv_colunas", nome, "critico", teste_colunas, nome)
    for coluna, tipo in SOMAS_CSV[nome].items():
        testar(f"csv_soma_{coluna}", nome, "critico", teste_soma, nome, coluna, tipo)
    testar("csv_encoding", nome, "critico", teste_encoding, nome)

## 2. Variação de linhas (alerta)
Compara a contagem atual de cada tabela com a da última execução registrada em `resultados_testes` (teste `contagem_linhas`).

- Sem execução anterior: `LINHA_DE_BASE`.
- A partir da segunda: `ALERTA` se a variação absoluta passar de 5%, senão `OK`.
- A contagem atual sempre vai para `valor`, e é a base da próxima execução. O teste não derruba o Job: a Gold é recarregada por
  completo a cada run (DEC-039), e uma variação grande pede conferência, não bloqueio.

In [ ]:
VARIACAO_MAXIMA = 0.05


def contagem_anterior(nome):
    r = (spark.table(TABELA_RESULTADOS)
        .filter((F.col("teste") == "contagem_linhas") & (F.col("tabela") == nome) & F.col("valor").isNotNull())
        .orderBy(F.col("executado_em").desc())
        .select("valor").first())
    return None if r is None else r["valor"]


def teste_variacao(nome):
    atual = contar_tabela(nome)
    anterior = contagem_anterior(nome)
    if anterior is None:
        return "LINHA_DE_BASE", atual, "sem execução anterior: contagem guardada como base"
    if anterior == 0:
        variacao = 0.0 if atual == 0 else float("inf")
    else:
        variacao = abs(atual - anterior) / anterior
    if variacao > VARIACAO_MAXIMA:
        return "ALERTA", atual, f"variação de {variacao:.1%} sobre a execução anterior"
    return "OK", atual, f"variação de {variacao:.1%} sobre a execução anterior"


for nome in TABELAS:
    testar("contagem_linhas", nome, "alerta", teste_variacao, nome)

## 3. Reconciliações entre tabelas (críticos)
Checks do contrato (`contracts/gold_schema.yaml`), conferidos na tabela gravada:

- `reconc_kpis_mensais_segmento`: `gold_kpis_mensais` = soma de `gold_kpis_segmento_mensal` por competência, em `kwh_faturado`,
  `receita_faturada` e `receita_energia`, exato (competência que falta de um dos lados também é divergência).
- `reconc_receita_total`: em `gold_reconciliacao_faturamento`, `receita_faturada_total = receita_faturada_cadastro +
  receita_faturada_orfas`, com tolerância de R$ 0,01 por linha.
- `reconc_decomposicao_pvm`: em `gold_receita_decomposicao`, `|efeito_preco + efeito_volume + efeito_mix − (receita_atual −
  receita_base)| <= R$ 0,025` por linha (DEC-014, DEC-029).

Valor nulo numa conta conta como divergência. `valor` = linhas divergentes.

In [ ]:
COLUNAS_KPIS = ["kwh_faturado", "receita_faturada", "receita_energia"]


def teste_kpis_mensais():
    segmento = (spark.table(f"{GOLD}.gold_kpis_segmento_mensal")
        .groupBy("competencia").agg(*[F.sum(c).alias(f"seg_{c}") for c in COLUNAS_KPIS]))
    mensais = spark.table(f"{GOLD}.gold_kpis_mensais").select("competencia", *COLUNAS_KPIS)
    juntos = mensais.join(segmento, "competencia", "full")
    r = juntos.agg(*[F.sum((~F.col(c).eqNullSafe(F.col(f"seg_{c}"))).cast("int")).alias(c)
                     for c in COLUNAS_KPIS]).first().asDict()
    divergentes = [c for c, n in r.items() if n]
    return ok_ou_falha(sum(r[c] for c in divergentes), f"competências com diferença em: {divergentes}")


def contar_fora(tabela, condicao_ok):
    return spark.table(f"{GOLD}.{tabela}").filter(f"NOT coalesce({condicao_ok}, false)").count()


def teste_receita_total():
    n = contar_fora("gold_reconciliacao_faturamento",
                    "abs(receita_faturada_total - (receita_faturada_cadastro + receita_faturada_orfas)) <= 0.01")
    return ok_ou_falha(n, "competências com total diferente de cadastro + órfãs")


def teste_decomposicao():
    n = contar_fora("gold_receita_decomposicao",
                    "abs(efeito_preco + efeito_volume + efeito_mix - (receita_atual - receita_base)) <= 0.025")
    return ok_ou_falha(n, "linhas com preço + volume + mix diferente da variação da receita")


testar("reconc_kpis_mensais_segmento", "gold_kpis_mensais", "critico", teste_kpis_mensais)
testar("reconc_receita_total", "gold_reconciliacao_faturamento", "critico", teste_receita_total)
testar("reconc_decomposicao_pvm", "gold_receita_decomposicao", "critico", teste_decomposicao)

## 4. Premissa da DEC-067 (crítico)
As UCs de teste do cadastro (UC-12, `cod_uc` com prefixo `TESTE`) ficam na `gold_dim_uc` (DEC-067). A decisão vale porque elas não
têm fatura, medidor, ocorrência, chamado, inspeção nem lugar na fila; por isso receita, energia, `n_uc`, DEC/FEC e fila não mudam.
`dec067_uc_teste_sem_linhas` confere, em cada tabela por UC, que nenhuma linha é de UC de teste (`valor` = linhas encontradas).

In [ ]:
TABELAS_POR_UC = ["gold_fato_faturamento_mensal", "gold_medidor_uc", "gold_interrupcoes_uc", "gold_atendimentos_uc",
                  "gold_inspecoes_uc", "gold_fila_suspeitas"]


def teste_uc_teste(nome):
    ucs_teste = spark.table(f"{GOLD}.gold_dim_uc").filter(F.col("cod_uc").like("TESTE%")).select("cod_uc")
    n = spark.table(f"{GOLD}.{nome}").join(ucs_teste, "cod_uc", "left_semi").count()
    return ok_ou_falha(n, "linhas de UC de teste (prefixo TESTE)")


for nome in TABELAS_POR_UC:
    testar("dec067_uc_teste_sem_linhas", nome, "critico", teste_uc_teste, nome)

## Resumo
Grava todos os resultados em `v_energy.qualidade.resultados_testes` (append, mesmo `execucao_id`) e imprime teste, tabela, tipo e
status, sem a coluna `valor`. Falha se algum teste `critico` tiver `FALHA`, listando os testes (o Job fica vermelho).

In [ ]:
resultados = spark.createDataFrame([tuple(r[c] for c in ESQUEMA_RESULTADOS.fieldNames()) for r in RESULTADOS],
                                   ESQUEMA_RESULTADOS)
resultados.write.format("delta").mode("append").saveAsTable(TABELA_RESULTADOS)
print(f"Execução {EXECUCAO_ID} | {EXECUTADO_EM.isoformat(timespec='seconds')} | gravada em {TABELA_RESULTADOS}")
resultados.select("teste", "tabela", "tipo", "status").show(len(RESULTADOS), truncate=False)

falhas = [f"{r['teste']} ({r['tabela']})" for r in RESULTADOS if r["tipo"] == "critico" and r["status"] == "FALHA"]
alertas = [f"{r['teste']} ({r['tabela']})" for r in RESULTADOS if r["status"] in ("ALERTA", "FALHA") and r["tipo"] == "alerta"]
if alertas:
    print(f"[ALERTA] conferir: {alertas}")
if falhas:
    raise Exception(f"Testes críticos com FALHA: {falhas}")
print("[OK] Nenhum teste crítico falhou.")